This notebook builds a two step pipeline for rice plants.

Step 1 is called Object Classification. It classifies the strain of the rice plant from an image.

Step 2 is called Object Detection. It detects and localizes sickness on the leaf of the rice plant, and counts how many lesions of each type are present.

The two models are trained separately so each one can be optimized for its own task. Strain classification and disease detection do not need to share the same model, and keeping them separate gives better accuracy for both.

The final result of the combined pipeline looks like this.

Strain: Jasmine | Health Alert: 3 Leaf Blast lesions detected

The notebook also includes ground truth comparisons and training graphs at the end of each step, so the accuracy of both models can be checked before combining them.

Install the libraries needed for both steps.

In [ ]:
!pip install fastai timm ultralytics opencv-python scikit-learn kagglehub --break-system-packages -q


Step 1: Strain Classification

The dataset used for this step is the Rice Image Dataset from Kaggle. It contains images of five rice varieties, Arborio, Basmati, Ipsala, Jasmine, and Karacadag, which is what strain classification is based on.

Rice Image Dataset
https://www.kaggle.com/datasets/muratkokludataset/rice-image-dataset

One thing to note about this dataset. The images are individual rice grains photographed on a plain background, not leaf or plant images captured in the field. This keeps the dataset small and the variety labels are accurate, but if the camera in the final setup is meant to photograph the growing plant rather than harvested grains, this model works best as a separate grain inspection step rather than being run on the same field photo as the disease detector in Step 2.

In [ ]:
import kagglehub

strain_dataset_path = kagglehub.dataset_download("muratkokludataset/rice-image-dataset")
print(f"Dataset downloaded to {strain_dataset_path}")


The dataset is organized as one folder per variety. Check the folder structure after downloading to confirm the path below points at the folder containing the five variety subfolders.

In [ ]:
from pathlib import Path
from fastai.vision.all import *

STRAIN_IMAGE_DIR = Path(strain_dataset_path) / "Rice_Image_Dataset"
IMG_SIZE = 224
BATCH_SIZE = 64
ARCH = "convnext_small_in22k"
EPOCHS_HEAD = 3
EPOCHS_FINE_TUNE = 8
MODEL_OUT = "strain_classifier.pkl"


This function builds the dataloaders used for training. It reads the variety label from each image's parent folder name. It is called twice, once with a small image size and once with a larger image size.

In [ ]:
def build_dataloaders(img_size, bs):
    dblock = DataBlock(
        blocks=(ImageBlock, CategoryBlock),
        get_items=get_image_files,
        get_y=parent_label,
        splitter=RandomSplitter(valid_pct=0.2, seed=42),
        item_tfms=Resize(img_size, method="squish"),
        batch_tfms=aug_transforms(size=img_size, min_scale=0.75) + [Normalize.from_stats(*imagenet_stats)],
    )
    return dblock.dataloaders(STRAIN_IMAGE_DIR, bs=bs)


Train on small images first. This trains quickly and confirms the pipeline works before spending time on the larger image size.

In [ ]:
dls = build_dataloaders(IMG_SIZE, BATCH_SIZE)
learn = vision_learner(dls, ARCH, metrics=error_rate).to_fp16()
learn.fine_tune(EPOCHS_HEAD, base_lr=3e-3)


Train again on larger images. This is called progressive resizing and it improves accuracy over training at one fixed size.

In [ ]:
dls_big = build_dataloaders(int(IMG_SIZE * 1.6), BATCH_SIZE // 2)
learn.dls = dls_big
learn.fine_tune(EPOCHS_FINE_TUNE, base_lr=1e-3)


Save the trained model.

In [ ]:
learn.export(MODEL_OUT)
print(f"Model saved to {MODEL_OUT}")


Evaluation. This section checks how accurate the strain classifier is by comparing its predictions against the ground truth labels, and shows the training curves and confusion matrix.

In [ ]:
learn.recorder.plot_loss()


In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(8, 8))


In [ ]:
preds, targs = learn.tta()
accuracy = (preds.argmax(dim=1) == targs).float().mean()
print(f"Validation accuracy with test time augmentation: {accuracy:.4f}")


This shows individual predictions next to the ground truth label, so the errors can be inspected directly.

In [ ]:
interp.plot_top_losses(9, nrows=3)


Step 2: Localized Disease Detection

Strain classification only tells us which variety a plant is. It cannot report something like three Leaf Blast lesions detected, because that requires locating each lesion on the leaf, not just labeling the whole image. This step trains a YOLO model for that purpose.

The Rice Image Dataset used in Step 1 has no bounding box annotations and is not leaf imagery, so a separate dataset is needed here. Two Kaggle datasets that already include bounding boxes for rice leaf diseases are listed below.

Rice Leaf Diseases, YOLO formatted
https://www.kaggle.com/datasets/yusufmurtaza01/rice-leaf-diseases

Rice Leaf Spot Disease Annotated Dataset
https://www.kaggle.com/datasets/hadiurrahmannabil/rice-leaf-spot-disease-annotated-dataset

Either dataset can be used as long as it is in YOLO format, meaning one label file per image with box coordinates and a class id. If the dataset is not already split into images, labels, and a data.yaml file, it needs to be reorganized into that structure before training.

Check GPU availability before training.

In [ ]:
!nvidia-smi


In [ ]:
lesion_dataset_path = kagglehub.dataset_download("yusufmurtaza01/rice-leaf-diseases")
print(f"Dataset downloaded to {lesion_dataset_path}")


Point this at the data.yaml file inside the downloaded dataset. The exact path depends on how the dataset is packaged, so check the folder structure after downloading.

In [ ]:
data_yaml_path = f"{lesion_dataset_path}/data.yaml"
CONF_THRESHOLD = 0.4


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

model.train(
    task="detect",
    mode="train",
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    patience=15,
    name="rice_lesion_detector",
)


Evaluation. Ultralytics runs validation on the held out set and reports precision, recall, and mAP against the ground truth boxes.

In [ ]:
metrics = model.val()
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")


The training run also saves graphs and a confusion matrix automatically. These display the loss curves, precision, recall, and mAP across all training epochs.

In [ ]:
from PIL import Image

results_dir = "runs/detect/rice_lesion_detector"
display(Image.open(f"{results_dir}/results.png"))
display(Image.open(f"{results_dir}/confusion_matrix.png"))


This shows a batch of validation images with the predicted boxes next to the ground truth boxes, so predictions can be checked visually.

In [ ]:
display(Image.open(f"{results_dir}/val_batch0_labels.jpg"))
display(Image.open(f"{results_dir}/val_batch0_pred.jpg"))


In [ ]:
from collections import Counter

def detect_lesions(model, image_path):
    results = model.predict(image_path, conf=CONF_THRESHOLD, imgsz=640, verbose=False)
    result = results[0]
    names = result.names
    counts = Counter()
    for box in result.boxes:
        counts[names[int(box.cls[0])]] += 1
    return dict(counts)

# best_model = YOLO(f"{results_dir}/weights/best.pt")
# counts = detect_lesions(best_model, "camera_frame.jpg")
# print(counts)


Combined Inference

This runs both models on one image and produces the final report. It requires the strain classifier from Step 1 and the trained lesion detector from Step 2.

Since Step 1 in this version works on individual grain images rather than field leaf photos, this combined step assumes two separate images are captured, one grain photo for strain and one leaf photo for disease, rather than a single shared frame.

In [ ]:
STRAIN_MODEL_PATH = "strain_classifier.pkl"
LESION_MODEL_PATH = f"{results_dir}/weights/best.pt"

strain_model = load_learner(STRAIN_MODEL_PATH)
lesion_model = YOLO(LESION_MODEL_PATH)


In [ ]:
def predict_strain(model, image_path):
    pred_class, pred_idx, probs = model.predict(image_path)
    return str(pred_class), float(probs[pred_idx])


def format_report(strain, lesion_counts):
    if not lesion_counts:
        return f"Strain: {strain} | Health Alert: No lesions detected"
    parts = [f"{count} {disease.replace('_', ' ').title()} lesions detected"
             for disease, count in lesion_counts.items()]
    return f"Strain: {strain} | Health Alert: {'; '.join(parts)}"


def analyze_frames(grain_image_path, leaf_image_path, strain_model, lesion_model):
    strain, confidence = predict_strain(strain_model, grain_image_path)
    lesion_counts = detect_lesions(lesion_model, leaf_image_path)
    report = format_report(strain, lesion_counts)
    return report, strain, confidence, lesion_counts


In [ ]:
grain_image_path = "grain_sample.jpg"
leaf_image_path = "leaf_sample.jpg"

report, strain, confidence, lesion_counts = analyze_frames(grain_image_path, leaf_image_path, strain_model, lesion_model)
print(report)
